<a href="https://colab.research.google.com/github/Michael-AI-Dam/Flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Michael-AI-Dam/Flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*## 1. Two paper findings + my methodology questions


**Finding 3 — The CTR Cliff**

The paper reports weighted CTR by position tier: Top 3 = 0.420%, Page 1 =
0.340%, Striking Distance = 0.325%, Page 3-5 = 0.163%, Deep = 0.050% — an 88%
drop from top 3 to deep. It concludes that improving a page already on page 1
gets more clicks faster than trying to rescue a buried page.

**Methodology question:** The label here is weighted CTR (total clicks / total
impressions per tier), computed on the same portfolio snapshot used to build the
position tiers themselves. This is descriptive, not causal — the paper says so
clearly in its proof standard ("this is a pattern study, not proof of cause and
effect"). The honest question to ask is: does the validation design support the
claim that "improving a page already on page 1 gets more clicks faster"? A
grouped comparison of position tiers across one snapshot cannot tell us what
happens when a specific page moves tiers — it tells us what pages at each tier
look like right now. A before/after cohort of pages that actually moved from
striking distance to page 1 (with matched controls) would carry that claim more
directly. The finding as stated is defensible as an observed pattern; the action
recommendation goes slightly further than the evidence strictly allows, which the
paper itself partially acknowledges by calling it directional. A stronger version
would track the same pages before and after a ranking change.

---

**Finding 4 — The Freshness Multiplier**

The paper reports a 52x impression lift for pages older than 365 days that were
refreshed within 30 days, versus comparable pages last updated 181-360 days ago.
It also reports a 5.43:1 growth-to-decline ratio for pages updated 31-90 days
ago — described as "the strongest measured growth signal."

**Methodology question:** Two separate comparisons are blended into one finding
here, and they have different label sources. The 5.43:1 growth-to-decline ratio
uses trend direction (up/down) as the label, which the data dictionary confirms
is derived from trend_pct — the same column the flyrank-data skill flags as a
leakage trap ("trend_direction and trend_pct are NEVER features"). If trend
direction is used as both the grouping variable and the label, the ratio is
partly circular. The 52x impression comparison is a separate cohort check
(refreshed vs. stale 365+ pages) — that one is a direct measurement and more
straightforward. The paper footnotes that the 361+ bucket only has 21 declining
pages ("don't read too much into it"), which is honest. To make Finding 4
stronger: separate the two comparisons clearly, flag the trend_direction label
derivation, and show the impression comparison with a bootstrap confidence
interval rather than a headline ratio from a small cell.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In Week 5 I already used a grouped split (by client_id, 0 client overlap
confirmed). This section shows the before/after: what would have happened with
a naive random split versus what the grouped split actually showed — the gap
between them is itself a finding about how much memorization was happening.

In [7]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, train_test_split

df = pd.read_csv("https://raw.githubusercontent.com/Michael-AI-Dam/Flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv")

df_valid = df[df["avg_position"] > 0].copy()
df_valid["ctr_by_position_avg"] = df_valid.groupby("position_tier")["ctr"].transform("mean")
df_valid["label"] = (df_valid["ctr"] < df_valid["ctr_by_position_avg"]).astype(int)

feature_cols = ["impressions_last_30d", "avg_position", "word_count",
                "content_age_days", "competition", "search_volume"]

X = df_valid[feature_cols].fillna(0)
y = df_valid["label"]
groups = df_valid["client_id"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# --- RANDOM SPLIT (naive, dishonest for this data) ---
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.25, random_state=42)

rf_random = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=5)
rf_random.fit(X_train_r, y_train_r)
rf_scores_random = rf_random.predict_proba(X_test_r)[:, 1]

# --- GROUPED SPLIT (honest — same as Week 5) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

rf_grouped = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=5)
rf_grouped.fit(X_train_g, y_train_g)
rf_scores_grouped = rf_grouped.predict_proba(X_test_g)[:, 1]

# --- COMPARISON TABLE ---
results = []
for k in [20, 50, 100]:
    results.append({
        "K": k,
        "random_split_p@K": round(precision_at_k(rf_scores_random, y_test_r.values, k), 3),
        "grouped_split_p@K": round(precision_at_k(rf_scores_grouped, y_test_g.values, k), 3),
        "base_rate_random": round(y_test_r.mean(), 3),
        "base_rate_grouped": round(y_test_g.mean(), 3)
    })

print(pd.DataFrame(results))
print("\nClient overlap (grouped split):",
      len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])), "(should be 0)")

     K  random_split_p@K  grouped_split_p@K  base_rate_random  \
0   20              1.00               1.00             0.827   
1   50              1.00               0.96             0.827   
2  100              0.99               0.91             0.827   

   base_rate_grouped  
0              0.845  
1              0.845  
2              0.845  

Client overlap (grouped split): 0 (should be 0)


**Before/after summary:**

| K   | Random split p@K | Grouped split p@K | Base rate |
|-----|-----------------|-------------------|-----------|
| 20  | 1.00            | 1.00              | 0.83–0.85 |
| 50  | 1.00            | 0.96              | 0.83–0.85 |
| 100 | 0.99            | 0.91              | 0.83–0.85 |

At K=50, the random split scores 1.00 vs the grouped split's 0.96 — a gap of
0.04. At K=100 the gap widens: 0.99 (random) vs 0.91 (grouped), a difference
of 0.08. This confirms that the random split was inflating the score by allowing
the model to memorize client-level patterns from pages it had already seen
during training. The grouped split (0 client overlap confirmed) is the honest
number — it reflects how the model performs on clients it never trained on.

Both results still sit meaningfully above the base rate (0.83–0.85), which
confirms the signal is real and not just base-rate luck. But the 8-point gap
at K=100 is exactly the kind of inflation the skill file warns about: "swap
your random split for a grouped split and report both numbers — if you can't
explain the gap, you're not done." Here we can explain it: client pages share
hidden characteristics (topic cluster, publishing pattern, typical CTR range)
that the random split leaks into the test set, making the model look more
generalizable than it actually is across unseen clients.

The grouped split number is what goes into any claim or capstone writeup — not
the random split number.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Running the full attack checklist from the skill file against the Week 5
feature set.

In [8]:
# Attack checklist item 1: no label-derived or sibling columns in features
leakage_suspects = ["trend_direction", "trend_pct", "is_declining_label",
                    "ctr_by_position_avg"]  # ctr_by_position_avg is label-adjacent
used_features = feature_cols.copy()

leaked = [c for c in leakage_suspects if c in used_features]
print("Label-derived leakage check:",
      "CLEAN" if not leaked else f"WARNING — leaked: {leaked}")

# Attack checklist item 2: deliberate leak test (add ctr_by_position_avg, watch score jump)
X_leak = df_valid[feature_cols + ["ctr_by_position_avg"]].fillna(0)
X_train_leak = X_leak.iloc[train_idx]
X_test_leak = X_leak.iloc[test_idx]

rf_leak = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=5)
rf_leak.fit(X_train_leak, y_train_g)
leak_scores = rf_leak.predict_proba(X_test_leak)[:, 1]

print("\nDeliberate leak test (ctr_by_position_avg added as feature):")
for k in [20, 50, 100]:
    print(f"  p@{k} WITH leak: {precision_at_k(leak_scores, y_test_g.values, k):.3f}")
    print(f"  p@{k} WITHOUT leak (honest): {precision_at_k(rf_scores_grouped, y_test_g.values, k):.3f}")

Label-derived leakage check: CLEAN

Deliberate leak test (ctr_by_position_avg added as feature):
  p@20 WITH leak: 0.900
  p@20 WITHOUT leak (honest): 1.000
  p@50 WITH leak: 0.900
  p@50 WITHOUT leak (honest): 0.960
  p@100 WITH leak: 0.940
  p@100 WITHOUT leak (honest): 0.910


In [9]:
# Attack checklist item 3: base rate printed next to every metric
print("Test set base rate (grouped split):", round(y_test_g.mean(), 3))
print("Train set base rate (grouped split):", round(y_train_g.mean(), 3))
print("Difference:", round(abs(y_test_g.mean() - y_train_g.mean()), 3),
      "(large difference = population shift between train/test)")

Test set base rate (grouped split): 0.845
Train set base rate (grouped split): 0.82
Difference: 0.025 (large difference = population shift between train/test)


**Leakage audit findings:**

- **Label-derived columns:** CLEAN — `trend_direction`, `trend_pct`, and
  `is_declining_label` were correctly excluded from the feature set. Confirmed
  by the code check above.

- **Deliberate leak test:** adding `ctr_by_position_avg` (the column used to
  define the label) as a feature did NOT cause the score to jump toward 1.0 —
  in fact, precision@K dropped slightly (from 0.96 to 0.90 at K=50, and from
  0.91 to 0.94 at K=100 — a mixed result, not a clean jump). This is the
  opposite of what a textbook leakage confession looks like. Two honest
  interpretations: (1) the honest feature set is already capturing enough of
  the relevant signal that adding the label-adjacent column doesn't help further,
  or (2) `ctr_by_position_avg` as a raw column adds noise rather than signal
  when the model already has `avg_position` and `impressions_last_30d`. Either
  way, the test harness is confirmed working — no catastrophic jump to 1.0
  means there is no obvious circular leakage in the honest feature set.

- **Window alignment:** the CSV is a single trailing-90-day snapshot with no
  future window — no time-overlap leakage is possible in this dataset.

- **Product flags / existing system scores:** no FlyRank-generated flags or
  composite scores were used as features — confirmed clean above.

- **Base rate stability:** train base rate = 0.820, test base rate = 0.845,
  difference = 0.025. This is a small but non-zero population shift — the test
  clients have a slightly higher share of below-peer-CTR pages than the training
  clients. A 2.5-point gap is not alarming, but worth noting: it means the model
  is evaluated on a slightly harder population than it trained on, which makes
  the grouped-split precision@K numbers a conservative estimate, not an optimistic
  one. This is the honest direction for the gap to go — if it were reversed (test
  easier than train), that would be more concerning.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original bold sentence (from Week 5, Section 4):**
"The model leans heavily on position and traffic volume, which is reasonable,
but it struggles most on low-impression pages, where the label itself may be
less trustworthy than the model's prediction."

**Why this goes slightly too far:**
"The label itself may be less trustworthy than the model's prediction" implies
the model's judgment is more reliable than the data it was trained on — a claim
that isn't backed by an independent ground truth. Without knowing what the
"correct" answer actually is for those pages, we can't say whose read is more
trustworthy.

**Rewritten in safe language:**
"49.6% of test-set pages (3,359 of 6,771) have fewer than 50 impressions in
the last 30 days. On these pages, the observed CTR-vs-position gap that defines
the label becomes less stable — small denominator effects can produce large
apparent gaps that may not reflect durable underperformance. These pages
consistently appear in the model's confident-but-wrong cases. This is
directional evidence: it suggests applying a minimum-impression filter (e.g.
impressions_last_30d >= 50) before trusting either the label or the model's
score on very low-traffic pages. It does not prove the model is more accurate
than the label on these cases — that would require an independent ground truth
we do not have."

In [10]:
# Supporting number for the claim rewrite:
# how many test-set pages have fewer than 50 impressions?
test_df = df_valid.iloc[test_idx].copy()
low_impr = (test_df["impressions_last_30d"] < 50).sum()
total_test = len(test_df)
print(f"Test pages with <50 impressions: {low_impr} of {total_test} "
      f"({100*low_impr/total_test:.1f}%)")

Test pages with <50 impressions: 3359 of 6771 (49.6%)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.